<a href="https://colab.research.google.com/github/mikadevonshire-design/codemender-oss/blob/main/MA_Test_MSIP_label.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#MAKE A COPY!!

# Install

In [ ]:
import os

# Auth

In [ ]:
! gcloud auth login


In [ ]:
# The temporary token is used to parse out [ , ], and ' characters
tmp_token = ! gcloud auth print-access-token
os.environ['access_token'] = str(str(str(tmp_token).replace("[","")).replace("]","")).replace("'","")

NameError: name 'os' is not defined

# Enter your variables







In [ ]:
project = "k8-and-storage-project" #@param {type:"string"}
location = "us" #@param {type:"string"}
# Create a new template using a unique name, or use an existing one
template = "agentspace-modelarmor-template" #@param {type:"string"}
# Copy these variables into the system env for use with bash commands
os.environ['project'] = project
os.environ['location'] = location
os.environ['template'] = template

# Create a new DLP Template if you want

In [ ]:
# CREATE A NEW DLP INSPECT TEMPLATE IF YOU WANT -BELOW IS TO ADD to an EXISTING

import os
import json

# Set the ID for your new inspection template
dlp_inspect_temp = "Agentspace-modelarmor" #@param {type:"string"}
# DEFINE YOUR DLP TEMPLATE
os.environ['dlp_inspect_temp'] = dlp_inspect_temp

# Construct the template configuration
msip_config = {
  "inspectTemplate": {
    "displayName": "MSIP Label Inspection Template",
    "inspectConfig": {
      "customInfoTypes": [
        {
          "infoType": {
            "name": "MSIP_Label"
          },
          "likelihood": "POSSIBLE",
          "metadataKeyValueExpression": {
            "keyRegex": "MSIP_Label_f64126a9-ff39-4d17-9984-3ebf4d55b46e_Enabled",
            "valueRegex": "true"
          }
        }
      ],
      "minLikelihood": "POSSIBLE"
    }
  },
  "templateId": os.environ['dlp_inspect_temp']
}

# Save the configuration to a file for the curl command
with open('msip_config.json', 'w') as f:
    json.dump(msip_config, f)

# Execute the POST request to create the template
!curl -X POST \
-H "Authorization: Bearer $access_token" \
-H "Content-Type: application/json" \
-H "X-Goog-User-Project: $project" \
-d @msip_config.json \
"https://dlp.googleapis.com/v2/projects/$project/locations/$location/inspectTemplates"


{
  "name": "projects/k8-and-storage-project/locations/us/inspectTemplates/msip-label-template",
  "displayName": "MSIP Label Inspection Template",
  "createTime": "2026-06-05T17:59:22.237999Z",
  "updateTime": "2026-06-05T17:59:22.237999Z",
  "inspectConfig": {
    "minLikelihood": "POSSIBLE",
    "limits": {},
    "customInfoTypes": [
      {
        "infoType": {
          "name": "MSIP_Label"
        },
        "likelihood": "POSSIBLE",
        "metadataKeyValueExpression": {
          "keyRegex": "MSIP_Label_f64126a9-ff39-4d17-9984-3ebf4d55b46e_Enabled",
          "valueRegex": "true"
        }
      }
    ]
  }
}


# Add the MSIPS detection to an existing template if you would like


In [ ]:

import os
import json
#ADD YOUR TEMPLATE
dlp_inspect_temp = "Agentspace-modelarmor" #@param {type:"string"}
# DEFINE YOUR DLP TEMPLATE
os.environ['dlp_inspect_temp'] = dlp_inspect_temp

# Define the updated configuration including existing infoTypes and the new MSIP label
update_config = {
  "inspectConfig": {
    "infoTypes": [
      { "name": "US_SOCIAL_SECURITY_NUMBER" },
      { "name": "US_HEALTHCARE_NPI" },
      { "name": "US_DRIVERS_LICENSE_NUMBER" },
      { "name": "CREDIT_CARD_DATA" },
      { "name": "CREDIT_CARD_NUMBER" }
    ],
    "customInfoTypes": [
      {
        "infoType": { "name": "MSIP_Label" },
        "likelihood": "POSSIBLE",
        "metadataKeyValueExpression": {
          "keyRegex": "MSIP_Label_f64126a9-ff39-4d17-9984-3ebf4d55b46e_Enabled",
          "valueRegex": "true"
        }
      }
    ],
    "minLikelihood": "POSSIBLE"
  }
}

# Save to a temporary file
with open('update_msip_config.json', 'w') as f:
    json.dump(update_config, f)

# Execute the PATCH request
# Note the use of ?updateMask=inspectConfig to specify only the config is changing
!curl -X PATCH \
-H "Authorization: Bearer $access_token" \
-H "Content-Type: application/json" \
-H "X-Goog-User-Project: $project" \
-d @update_msip_config.json \
"https://dlp.googleapis.com/v2/projects/$project/locations/$location/inspectTemplates/$dlp_inspect_temp?updateMask=inspectConfig"


# Upload the MSIPS File

In [ ]:
# Please name your file example.pdf before uploading
from google.colab import files
uploaded = files.upload()

Saving setest_doc_mp7.docx to setest_doc_mp7.docx


# Test to see if MSIPS file is detected by Model Armor

In [ ]:
!echo '{userPromptData: {byteItem: {byteDataType: "WORD_DOCUMENT", byteData: "'$(base64 -w 0 '/content/setest_doc_mp7.docx')'"}}}' | curl -X POST -d @- \
-H 'Content-Type: application/json' \
-H "Authorization: Bearer $access_token" \
"https://modelarmor.us.rep.googleapis.com/v1alpha/projects/$project/locations/us/templates/$template:sanitizeUserPrompt"


{
  "sanitizationResult": {
    "filterMatchState": "MATCH_FOUND",
    "filterResults": {
      "csam": {
        "csamFilterFilterResult": {
          "executionState": "EXECUTION_SUCCESS",
          "matchState": "NO_MATCH_FOUND"
        }
      },
      "malicious_uris": {
        "maliciousUriFilterResult": {
          "executionState": "EXECUTION_SUCCESS",
          "matchState": "NO_MATCH_FOUND"
        }
      },
      "rai": {
        "raiFilterResult": {
          "executionState": "EXECUTION_SUCCESS",
          "matchState": "NO_MATCH_FOUND",
          "raiFilterTypeResults": {
            "sexually_explicit": {
              "matchState": "NO_MATCH_FOUND"
            },
            "hate_speech": {
              "matchState": "NO_MATCH_FOUND"
            },
            "harassment": {
              "matchState": "NO_MATCH_FOUND"
            },
            "dangerous": {
              "matchState": "NO_MATCH_FOUND"
            }
          }
        }
      },
      "pi_and_ja

# JUST FYI MY Model Armor Template

In [ ]:
#MA template details
!curl -X GET \
-H "Authorization: Bearer $access_token" \
"https://modelarmor.$location.rep.googleapis.com/v1alpha/projects/$project/locations/$location/templates/$template"


{
  "name": "projects/k8-and-storage-project/locations/us/templates/agentspace-modelarmor-template",
  "createTime": "2025-08-11T14:27:10.493636176Z",
  "updateTime": "2026-03-16T18:19:19.194210976Z",
  "filterConfig": {
    "raiSettings": {
      "raiFilters": [
        {
          "filterType": "HATE_SPEECH",
          "confidenceLevel": "MEDIUM_AND_ABOVE"
        },
        {
          "filterType": "DANGEROUS",
          "confidenceLevel": "MEDIUM_AND_ABOVE"
        },
        {
          "filterType": "SEXUALLY_EXPLICIT",
          "confidenceLevel": "MEDIUM_AND_ABOVE"
        },
        {
          "filterType": "HARASSMENT",
          "confidenceLevel": "MEDIUM_AND_ABOVE"
        }
      ]
    },
    "sdpSettings": {
      "advancedConfig": {
        "inspectTemplate": "projects/k8-and-storage-project/locations/us/inspectTemplates/Agentspace-modelarmor",
        "deidentifyTemplate": "projects/k8-and-storage-project/locations/us/deidentifyTemplates/Agentspace-modelarmor-deiden"


# JUST FYI MY DLP TEMPLATE

In [ ]:
!curl -X GET \
-H "Authorization: Bearer $access_token" \
-H "Content-Type: application/json" \
-H "X-Goog-User-Project: $project" \
"https://dlp.googleapis.com/v2/projects/$project/locations/$location/inspectTemplates/$dlp_inspect_temp"



{
  "name": "projects/k8-and-storage-project/locations/us/inspectTemplates/Agentspace-modelarmor",
  "createTime": "2025-08-11T14:36:56.694351Z",
  "updateTime": "2026-06-04T23:46:36.803039Z",
  "inspectConfig": {
    "infoTypes": [
      {
        "name": "US_SOCIAL_SECURITY_NUMBER"
      },
      {
        "name": "US_HEALTHCARE_NPI"
      },
      {
        "name": "US_DRIVERS_LICENSE_NUMBER"
      },
      {
        "name": "CREDIT_CARD_DATA"
      },
      {
        "name": "CREDIT_CARD_NUMBER"
      }
    ],
    "minLikelihood": "POSSIBLE",
    "limits": {},
    "customInfoTypes": [
      {
        "infoType": {
          "name": "MSIP_Label"
        },
        "likelihood": "POSSIBLE",
        "sensitivityScore": {
          "score": "SENSITIVITY_HIGH"
        },
        "metadataKeyValueExpression": {
          "keyRegex": "MSIP_Label_f64126a9-ff39-4d17-9984-3ebf4d55b46e_Enabled",
          "valueRegex": "true"
        }
      }
    ]
  }
}
